In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill

from src.metric import *

In [ ]:
bi_ndcg_list = []
cross_ndcg_list= []

bi_spearman_list = []
cross_spearman_list = []

bi_topk_list = []
cross_topk_list = []

bi_map_list=[]
cross_map_list=[]

bi_mrr_list=[]
cross_mrr_list=[]

val_resume_texts=val_df['resume_text'].tolist()

In [ ]:
for jd,group in val_df.groupby('job_description_text'):

    if len(group)<2:
        continue
        
    group_label=group[['resume_text','label']].drop_duplicates()
    label_map=dict(zip(group_label['resume_text'],group_label['label']))

    # Stage 1 — Bi-Encoder retrieval (top-50)

    jd_emb=bi_encoder.encode(jd,convert_to_tensor=True)
    
    hits=util.semantic_search(jd_emb, val_resume_emb, top_k=4)
    hits=hits[0]

    retrieval_rows = []
    for hit in hits:
        idx=hit["corpus_id"]
        resume_text=val_resume_texts[idx]
        label=label_map.get(resume_text, None)
        if label is None:
            continue
        retrieval_rows.append({
            "val_resume_text":resume_text,
            "score": float(hit["score"]),
            "label": label
        })
    retrieval_df=pd.DataFrame(retrieval_rows)

    #Ndcg
    bi_ndcg=ndcg_metric(retrieval_df)
    if bi_ndcg not None:
        bi_ndcg_list.append(bi_ndcg)
    
    # Spearman
    bi_corr=corr_metric(retrieval_df)
    if bi_corr not None:
        bi_spearman_list.append(bi_corr)

    # Top-K accuracy
    bi_topk=topk_metric(retrieval_df)
    if bi_topk not None:
        bi_topk_list.append(bi_topk)

    #Mrr
    bi_mrr=mrr_metric(retrieval_df)
    if bi_mrr not None:
        bi_mrr_list.append(bi_mrr)
    
    #Mrr
    bi_map=map_metric(retrieval_df)
    if bi_map not None:
        bi_map_list.append(bi_map)
    
            
    # Stage 2 — Cross-Encoder reranking (on bi-encoder top-50)

    pairs=list(zip([jd]*len(retrieval_df),retrieval_df['val_resume_text']))

    cross_scores=cross_encoder_model.predict(pairs,batch_size=16)

    rerank_df=retrieval_df.copy()
    rerank_df['score']=cross_scores
    rerank_df=rerank_df.sort_values("score",ascending=False).reset_index(drop=True)


    #Ndcg
    cross_ndcg=ndcg_metric(rerank_df)
    if cross_ndcg not None:
        cross_ndcg_list.append(cross_ndcg)
    
    # Spearman
    cross_corr=corr_metric(rerank_df)
    if cross_corr not None:
        cross_spearman_list.append(cross_corr)

    # Top-K accuracy
    cross_topk=topk_metric(rerank_df)
    if cross_topk not None:
        cross_topk_list.append(cross_topk)

    #Mrr
    cross_mrr=mrr_metric(rerank_df)
    if cross_mrr not None:
        cross_mrr_list.append(cross_mrr)
    
    #Mrr
    cross_map=map_metric(rerank_df)
    if cross_map not None:
        cross_map_list.append(cross_map)
        
            

    

In [ ]:
    
print("\n" + "=" * 60)
print("FINAL TWO-STAGE RETRIEVAL RESULTS")

print("=" * 45)
print(f"{'Metric':<25} {'Bi-Encoder':>8}   {'Cross-Encoder':>13}")
print("=" * 45)
print(f"{'NDCG@10':<25} {np.mean(bi_ndcg_list):>8.4f}   {np.mean(cross_ndcg_list):>13.4f}")
print(f"{'Spearman ':<25} {np.mean(bi_spearman_list):>8.4f}   {np.mean(cross_spearman_list):>13.4f}")
print(f"{'Top-{3} Accuracy':<25} {np.mean(bi_topk_list):>8.4f}   {np.mean(cross_topk_list):>13.4f}")
print(f"{'MRR':<25} {np.mean(bi_mrr_list):>8.4f}   {np.mean(cross_mrr_list):>13.4f}")
print(f"{'MAP':<25} {np.mean(bi_map_list):>8.4f}   {np.mean(cross_map_list):>13.4f}")
print("=" * 45)